## Setup and Imports

In [42]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from tqdm import tqdm
from collections import Counter

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


## Define Relation Labels and Configuration

In [43]:
import re

LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}

def norm_ent(label: str) -> str:
    if label is None:
        return ""
    lab = str(label).strip()
    if lab.lower() == "ddf":
        return "DDF"
    return lab

def norm_span(s: str) -> str:
    # consigliato per ridurre mismatch banali sugli span
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

In [44]:
# Define legal relation predicates
RELATION_LABELS = [
    "no relation",  # For negative samples
    "administered",
    "affect",
    "change abundance",
    "change effect",
    "change expression",
    "compared to",
    "impact",
    "influence",
    "interact",
    "is a",
    "is linked to",
    "located in",
    "part of",
    "produced by",
    "strike",
    "target",
    "used by"
]

label2id = {label: idx for idx, label in enumerate(RELATION_LABELS)}
id2label = {idx: label for idx, label in enumerate(RELATION_LABELS)}

print(f"Total relation labels: {len(RELATION_LABELS)}")
print(f"Labels: {RELATION_LABELS}")

# Define legal entity type relations (subject_label, predicate, object_label)
# Order matters: relation is from subject to object
LEGAL_RELATIONS = [
    ("DDF", "affect", "DDF"),
    ("microbiome", "is linked to", "DDF"),
    ("DDF", "target", "human"),
    ("drug", "change effect", "DDF"),
    ("DDF", "is a", "DDF"),
    ("microbiome", "located in", "human"),
    ("chemical", "influence", "DDF"),
    ("dietary supplement", "influence", "DDF"),
    ("DDF", "target", "animal"),
    ("chemical", "impact", "microbiome"),
    ("anatomical location", "located in", "animal"),
    ("microbiome", "located in", "animal"),
    ("chemical", "located in", "anatomical location"),
    ("bacteria", "part of", "microbiome"),
    ("DDF", "strike", "anatomical location"),
    ("drug", "administered", "animal"),
    ("bacteria", "influence", "DDF"),
    ("drug", "impact", "microbiome"),
    ("DDF", "change abundance", "microbiome"),
    ("microbiome", "located in", "anatomical location"),
    ("microbiome", "used by", "biomedical technique"),
    ("chemical", "produced by", "microbiome"),
    ("dietary supplement", "impact", "microbiome"),
    ("bacteria", "located in", "animal"),
    ("animal", "used by", "biomedical technique"),
    ("chemical", "impact", "bacteria"),
    ("chemical", "located in", "animal"),
    ("food", "impact", "bacteria"),
    ("microbiome", "compared to", "microbiome"),
    ("human", "used by", "biomedical technique"),
    ("bacteria", "change expression", "gene"),
    ("chemical", "located in", "human"),
    ("drug", "interact", "chemical"),
    ("food", "administered", "human"),
    ("DDF", "change abundance", "bacteria"),
    ("chemical", "interact", "chemical"),
    ("chemical", "part of", "chemical"),
    ("dietary supplement", "impact", "bacteria"),
    ("DDF", "interact", "chemical"),
    ("food", "impact", "microbiome"),
    ("food", "influence", "DDF"),
    ("bacteria", "located in", "human"),
    ("dietary supplement", "administered", "human"),
    ("bacteria", "interact", "chemical"),
    ("drug", "change expression", "gene"),
    ("drug", "impact", "bacteria"),
    ("drug", "administered", "human"),
    ("anatomical location", "located in", "human"),
    ("dietary supplement", "change expression", "gene"),
    ("chemical", "change expression", "gene"),
    ("bacteria", "interact", "bacteria"),
    ("drug", "interact", "drug"),
    ("microbiome", "change expression", "gene"),
    ("bacteria", "interact", "drug"),
    ("food", "change expression", "gene")
]

# Create lookup structures for legal relations
# Map (subject_label, object_label) -> set of predicates
legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    s = norm_ent(s); o = norm_ent(o)
    legal_pairs.setdefault((s, o), set()).add(p)

print(f"\nTotal legal relation patterns: {len(LEGAL_RELATIONS)}")
print(f"Total unique entity type pairs: {len(legal_pairs)}")

# Configuration
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"  # BioBERT for biomedical text
output_model_dir = "models/bert_biomedbert_re"
max_length = 512
NEGATIVE_SAMPLE_MULTIPLIER = 5  # Number of negative samples per positive sample

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")
print(f"Negative sample multiplier: {NEGATIVE_SAMPLE_MULTIPLIER}")

Total relation labels: 18
Labels: ['no relation', 'administered', 'affect', 'change abundance', 'change effect', 'change expression', 'compared to', 'impact', 'influence', 'interact', 'is a', 'is linked to', 'located in', 'part of', 'produced by', 'strike', 'target', 'used by']

Total legal relation patterns: 55
Total unique entity type pairs: 52

Model: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Output directory: models/bert_biomedbert_re
Negative sample multiplier: 5


## BERT Model with Entity Markers

In [45]:
class BertForREWithEntityMarkers(nn.Module):
    """
    BERT model for Relation Extraction with entity marker tokens.
    
    The model extracts hidden states at [E1] and [E2] token positions,
    concatenates them, and passes through a classification head.
    """
    
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        
        # Classification head: concatenated entity representations -> labels
        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Linear(hidden_size * 2, num_labels)
        
        self.num_labels = num_labels
    
    def forward(self, input_ids, attention_mask, e1_mask, e2_mask, labels=None):
        """
        Args:
            input_ids: Token IDs [batch_size, seq_len]
            attention_mask: Attention mask [batch_size, seq_len]
            e1_mask: Mask for [E1] token position [batch_size, seq_len]
            e2_mask: Mask for [E2] token position [batch_size, seq_len]
            labels: Ground truth labels [batch_size]
        """
        # Get BERT outputs
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        sequence_output = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]
        
        # Extract hidden states at [E1] and [E2] positions
        # e1_mask and e2_mask are one-hot vectors indicating token positions
        e1_h = torch.bmm(e1_mask.unsqueeze(1).float(), sequence_output).squeeze(1)  # [batch_size, hidden_size]
        e2_h = torch.bmm(e2_mask.unsqueeze(1).float(), sequence_output).squeeze(1)  # [batch_size, hidden_size]
        
        # Concatenate entity representations
        concat_h = torch.cat([e1_h, e2_h], dim=-1)  # [batch_size, hidden_size * 2]
        concat_h = self.dropout(concat_h)
        
        # Classification
        logits = self.classifier(concat_h)  # [batch_size, num_labels]
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        return {
            'loss': loss,
            'logits': logits
        }


print("BERT RE model class defined")

BERT RE model class defined


## Data Loading Functions

In [46]:
def load_re_data(file_paths):
    """Load relation extraction data from multiple JSON files."""
    all_data = {}
    
    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_data.update(data)
            print(f"Loaded {len(data)} documents from {os.path.basename(file_path)}")
        else:
            print(f"Warning: {file_path} not found")
    
    return all_data


print("Data loading function defined")

Data loading function defined


## Load Training and Dev Data

In [47]:
# Load training data from three quality levels
train_files = [
    "../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/json_format/train_gold.json",
    "../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver.json",
    "../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/bronze_quality/json_format/train_bronze.json",
    # opzionale (se vuoi includerlo):
    "../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver_2025.json",
]

train_data = load_re_data(train_files)
print(f"\nTotal training documents: {len(train_data)}")

Loaded 639 documents from train_gold.json
Loaded 811 documents from train_silver.json
Loaded 2972 documents from train_bronze.json
Loaded 499 documents from train_silver_2025.json

Total training documents: 4921


In [48]:
# Load dev data
dev_data = load_re_data([
    "../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json"
])
print(f"Total dev documents: {len(dev_data)}")

Loaded 80 documents from dev.json
Total dev documents: 80


## Prepare Relation Extraction Examples

For each document:
1. Extract positive relation examples from annotations
2. Generate negative examples by pairing entities that are NOT related
3. Apply the negative sample multiplier to balance the dataset

In [49]:
from collections import defaultdict
def create_full_text_with_offsets(title, abstract):
    """
    Create full text by concatenating title and abstract.
    Returns full text and offset for abstract entities.
    """
    full_text = f"{title} {abstract}"
    abstract_offset = len(title) + 1
    return full_text, abstract_offset


def adjust_entity_positions(entity, abstract_offset):
    """
    Adjust entity character positions to account for title + abstract concatenation.
    """
    if entity['location'] == 'abstract':
        return {
            'start_idx': entity['start_idx'] + abstract_offset,
            'end_idx': entity['end_idx'] + abstract_offset,
            'text_span': entity['text_span'],
            'label': entity['label']
        }
    else:
        return {
            'start_idx': entity['start_idx'],
            'end_idx': entity['end_idx'],
            'text_span': entity['text_span'],
            'label': entity['label']
        }


#TODO CHANGE
MAX_PAIR_CHARS = 400  # prova 300/400/500

def char_distance(a, b):
    # distanza tra due mention (start inclusive)
    return abs(a["start_idx"] - b["start_idx"])
def prepare_re_examples(data, negative_multiplier=1, legal_pairs=None):
    """
    Prepare relation extraction examples with positive and negative samples.
    Only considers entity pairs that match legal relation patterns.

    Args:
        data: Dictionary of documents with entities and mention_level_relations
        negative_multiplier: Number of negative samples per positive sample
        legal_pairs: Dict mapping (subject_label, object_label) -> set(predicates)

    Returns:
        List of examples: {text, subject, object, predicate, pmid}
    """
    def loc_rank(loc: str) -> int:
        return 0 if loc == "title" else 1  # title preferred over abstract

    def best_pair(subj_cands, obj_cands):
        """
        Choose the best (subject, object) mention pair among duplicates.
        Preference:
          1) title-title > title-abstract > abstract-abstract
          2) same location preferred
          3) minimal distance in text
        """
        best = None
        best_score = None

        for s in subj_cands:
            for o in obj_cands:
                # avoid identical mention used as both
                if s["start_idx"] == o["start_idx"] and s["end_idx"] == o["end_idx"] and s["location"] == o["location"]:
                    continue

                loc_combo = loc_rank(s["location"]) + loc_rank(o["location"])
                same_loc = 0 if s["location"] == o["location"] else 1
                dist = abs(s["start_idx"] - o["start_idx"])
                score = (loc_combo, same_loc, dist)

                if best_score is None or score < best_score:
                    best_score = score
                    best = (s, o)

        return best

    examples = []

    for pmid, article in tqdm(data.items(), desc="Preparing RE examples"):
        title = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abstract_offset = create_full_text_with_offsets(title, abstract)

        entities = article["entities"]
        relations = article.get("mention_level_relations", [])

        # normalize + adjust offsets
        adjusted_entities = [
            {
                **adjust_entity_positions(e, abstract_offset),
                "label": norm_ent(e["label"]),
                "text_span": norm_span(e["text_span"]),
                "location": e["location"],
            }
            for e in entities
        ]

        # index for (span,label) -> list of mentions
        ent_index = defaultdict(list)
        for e in adjusted_entities:
            ent_index[(e["text_span"], e["label"])].append(e)

        # -------- positives --------
        positive_pairs = set()

        for relation in relations:
            subj_text = norm_span(relation["subject_text_span"])
            obj_text  = norm_span(relation["object_text_span"])
            subj_lab  = norm_ent(relation["subject_label"])
            obj_lab   = norm_ent(relation["object_label"])
            pred      = relation["predicate"].strip()

            if pred not in LEGAL_RELATION_LABELS:
                continue
            if subj_lab not in LEGAL_ENTITY_LABELS or obj_lab not in LEGAL_ENTITY_LABELS:
                continue

            subj_cands = ent_index.get((subj_text, subj_lab), [])
            obj_cands  = ent_index.get((obj_text, obj_lab), [])

            pair = best_pair(subj_cands, obj_cands)
            if not pair:
                continue

            subject, obj = pair

            # optional safety: keep only legal type-pairs if provided
            type_pair = (subject["label"], obj["label"])
            if legal_pairs is not None and type_pair not in legal_pairs:
                continue

            examples.append({
                "text": full_text,
                "subject": subject,
                "object": obj,
                "predicate": pred,
                "pmid": pmid,
            })

            pair_key = (subject["start_idx"], subject["end_idx"], obj["start_idx"], obj["end_idx"])
            positive_pairs.add(pair_key)

        # -------- negatives --------
        num_negatives = len(positive_pairs) * negative_multiplier
        negative_candidates = []

        if num_negatives > 0:
            for i, subj in enumerate(adjusted_entities):
                for j, obj in enumerate(adjusted_entities):
                    if i == j:
                        continue

                    type_pair = (subj["label"], obj["label"])
                    if legal_pairs is not None and type_pair not in legal_pairs:
                        continue
                    if char_distance(subj, obj) > MAX_PAIR_CHARS:
                        continue
                    pair_key = (subj["start_idx"], subj["end_idx"], obj["start_idx"], obj["end_idx"])
                    if pair_key in positive_pairs:
                        continue

                    negative_candidates.append({
                        "text": full_text,
                        "subject": subj,
                        "object": obj,
                        "predicate": "no relation",
                        "pmid": pmid,
                    })

            if negative_candidates:
                num_to_sample = min(num_negatives, len(negative_candidates))
                examples.extend(random.sample(negative_candidates, num_to_sample))

    return examples


In [50]:
# Prepare training examples
print("Preparing training examples...")
train_examples = prepare_re_examples(train_data, negative_multiplier=NEGATIVE_SAMPLE_MULTIPLIER, legal_pairs=legal_pairs)

# Count positive vs negative
positive_count = sum(1 for ex in train_examples if ex['predicate'] != 'no relation')
negative_count = sum(1 for ex in train_examples if ex['predicate'] == 'no relation')

print(f"\nTraining examples prepared: {len(train_examples)}")
print(f"  Positive examples: {positive_count}")
print(f"  Negative examples: {negative_count}")
print(f"  Ratio (neg/pos): {negative_count/positive_count:.2f}")

Preparing training examples...


Preparing RE examples: 100%|██████████| 4921/4921 [00:03<00:00, 1268.33it/s]



Training examples prepared: 319650
  Positive examples: 53791
  Negative examples: 265859
  Ratio (neg/pos): 4.94


In [51]:
# Prepare dev examples
print("Preparing dev examples...")
dev_examples = prepare_re_examples(dev_data, negative_multiplier=NEGATIVE_SAMPLE_MULTIPLIER, legal_pairs=legal_pairs)

positive_count_dev = sum(1 for ex in dev_examples if ex['predicate'] != 'no relation')
negative_count_dev = sum(1 for ex in dev_examples if ex['predicate'] == 'no relation')

print(f"\nDev examples prepared: {len(dev_examples)}")
print(f"  Positive examples: {positive_count_dev}")
print(f"  Negative examples: {negative_count_dev}")

Preparing dev examples...


Preparing RE examples: 100%|██████████| 80/80 [00:00<00:00, 1250.78it/s]


Dev examples prepared: 6580
  Positive examples: 1116
  Negative examples: 5464


In [11]:
# Show example
print("\nExample training instance:")
example = train_examples[0]
print(f"  Text: {example['text'][:150]}...")
print(f"  Subject: '{example['subject']['text_span']}' [{example['subject']['label']}]")
print(f"  Object: '{example['object']['text_span']}' [{example['object']['label']}]")
print(f"  Predicate: {example['predicate']}")


Example training instance:
  Text: Probiotics and microbial metabolites maintain barrier and neuromuscular functions and clean protein aggregation to delay disease progression in TDP43 ...
  Subject: 'α-SMA' [chemical]
  Object: 'colon' [anatomical location]
  Predicate: located in


In [52]:
print("train_docs:", len(train_data))
print("dev_docs:", len(dev_data))

print("train_examples:", len(train_examples))
print("dev_examples:", len(dev_examples))

pos = sum(1 for ex in train_examples if ex["predicate"] != "no relation")
neg = len(train_examples) - pos
print("train_pos:", pos, "train_neg:", neg, "neg/pos:", neg/max(pos,1))

train_docs: 4921
dev_docs: 80
train_examples: 319650
dev_examples: 6580
train_pos: 53791 train_neg: 265859 neg/pos: 4.9424439032551915


## Initialize Tokenizer and Add Special Tokens

In [53]:
# Initialize tokenizer
print("Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Add special entity marker tokens
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
tokenizer.add_special_tokens(special_tokens)

# Get token IDs for entity markers
e1_token_id = tokenizer.convert_tokens_to_ids("[E1]")
e2_token_id = tokenizer.convert_tokens_to_ids("[E2]")

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"  Vocabulary size (with special tokens): {len(tokenizer)}")
print(f"  [E1] token ID: {e1_token_id}")
print(f"  [E2] token ID: {e2_token_id}")

Initializing tokenizer...
Tokenizer loaded: BertTokenizer
  Vocabulary size (with special tokens): 30526
  [E1] token ID: 30522
  [E2] token ID: 30524


## Tokenization with Entity Markers

In [54]:
def insert_entity_markers(text, subject, obj):
    """
    Insert entity marker tokens around subject and object entities.
    
    Args:
        text: Full text
        subject: Subject entity dict with start_idx, end_idx
        obj: Object entity dict with start_idx, end_idx
    
    Returns:
        Text with markers inserted
    """
    # Sort entities by position to insert markers correctly
    entities = [(subject['start_idx'], subject['end_idx'], '[E1]', '[/E1]'),
                (obj['start_idx'], obj['end_idx'], '[E2]', '[/E2]')]
    entities = sorted(entities, key=lambda x: x[0])
    
    # Insert markers from right to left to maintain positions
    marked_text = text
    offset = 0
    
    for start, end, start_marker, end_marker in entities:
        # Adjust positions with offset
        adj_start = start + offset
        adj_end = end + offset + 1  # +1 because end_idx is inclusive
        
        # Insert markers
        marked_text = (marked_text[:adj_start] + start_marker + 
                      marked_text[adj_start:adj_end] + end_marker + 
                      marked_text[adj_end:])
        
        # Update offset
        offset += len(start_marker) + len(end_marker)
    
    return marked_text


import re

def build_window_around_entities(text, subject, obj, window_chars=300):
    """
    Build a substring window around subject+object to avoid truncation.
    Recomputes subject/object offsets within the window.
    Assumes start/end are inclusive in the original text.
    """
    s_start, s_end = subject["start_idx"], subject["end_idx"]
    o_start, o_end = obj["start_idx"], obj["end_idx"]

    left = min(s_start, o_start)
    right = max(s_end, o_end)

    # expand window
    win_start = max(0, left - window_chars)
    win_end = min(len(text) - 1, right + window_chars)  # inclusive

    window_text = text[win_start:win_end + 1]

    # shift entity indices into window coordinates
    subj_w = dict(subject)
    obj_w = dict(obj)

    subj_w["start_idx"] = s_start - win_start
    subj_w["end_idx"] = s_end - win_start
    obj_w["start_idx"] = o_start - win_start
    obj_w["end_idx"] = o_end - win_start

    # safety clamp
    for ent in (subj_w, obj_w):
        ent["start_idx"] = max(0, min(ent["start_idx"], len(window_text) - 1))
        ent["end_idx"] = max(0, min(ent["end_idx"], len(window_text) - 1))

    return window_text, subj_w, obj_w


def tokenize_re_example(
    example,
    tokenizer,
    e1_token_id,
    e2_token_id,
    max_length=512,
    window_chars=300,
    fallback_to_fulltext=True,
):
    """
    Tokenize RE example using a window around entities so marker tokens are not truncated.
    """
    # 1) window around entities (recommended)
    w_text, w_subj, w_obj = build_window_around_entities(
        example["text"], example["subject"], example["object"], window_chars=window_chars
    )
    marked_text = insert_entity_markers(w_text, w_subj, w_obj)

    # 2) tokenize
    encoding = tokenizer(
        marked_text,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt",
    )


    input_ids = encoding["input_ids"].squeeze(0)
    attention_mask = encoding["attention_mask"].squeeze(0)

    e1_mask = (input_ids == e1_token_id).long()
    e2_mask = (input_ids == e2_token_id).long()

    # 3) if markers got lost (rare), optionally fallback to full text,
    #    otherwise raise/skip upstream
    if (e1_mask.sum().item() != 1 or e2_mask.sum().item() != 1) and fallback_to_fulltext:
        marked_text = insert_entity_markers(example["text"], example["subject"], example["object"])
        if marked_text.count("[E1]") != 1 or marked_text.count("[E2]") != 1:
            return None
        encoding = tokenizer(
            marked_text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        e1_mask = (input_ids == e1_token_id).long()
        e2_mask = (input_ids == e2_token_id).long()

    # 4) final check: if markers missing, skip example
    if e1_mask.sum().item() != 1 or e2_mask.sum().item() != 1:
        if example.get("pmid") == "38606018" or example.get("pmid") == 38606018:
            print("BAD MARKERS DEBUG pmid", example.get("pmid"))
            print("subj", example["subject"]["text_span"], example["subject"]["start_idx"], example["subject"]["end_idx"])
            print("obj ", example["object"]["text_span"], example["object"]["start_idx"], example["object"]["end_idx"])
        return None


    label = label2id[example["predicate"]]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "e1_mask": e1_mask,
        "e2_mask": e2_mask,
        "labels": torch.tensor(label, dtype=torch.long),
    }


print("Tokenization functions defined")

Tokenization functions defined


In [55]:
# Test tokenization
test_example = train_examples[0]
tokenized = tokenize_re_example(test_example, tokenizer, e1_token_id, e2_token_id)

print("Test tokenization:")
print(f"  Input IDs shape: {tokenized['input_ids'].shape}")
print(f"  E1 mask sum (should be 1): {tokenized['e1_mask'].sum().item()}")
print(f"  E2 mask sum (should be 1): {tokenized['e2_mask'].sum().item()}")
print(f"  Label: {tokenized['labels'].item()} ({id2label[tokenized['labels'].item()]})")

# Show marked text
marked = insert_entity_markers(test_example['text'], test_example['subject'], test_example['object'])
print(f"\nMarked text preview: {marked[:200]}...")

Test tokenization:
  Input IDs shape: torch.Size([146])
  E1 mask sum (should be 1): 1
  E2 mask sum (should be 1): 1
  Label: 12 (located in)

Marked text preview: Probiotics and microbial metabolites maintain barrier and neuromuscular functions and clean protein aggregation to delay disease progression in TDP43 mutation mice. Amyotrophic lateral sclerosis (ALS)...


## Create Dataset Class
### Pre-tokenize once + tensor-only dataset (with disk cache)

In [15]:
import torch
from dataclasses import dataclass
from transformers import PreTrainedTokenizerBase

@dataclass
class REDataCollatorWithPadding:
    tokenizer: PreTrainedTokenizerBase
    pad_to_multiple_of: int | None = None

    def __call__(self, features):
        # features: list of dict {input_ids, attention_mask, e1_mask, e2_mask, labels}
        labels = torch.stack([f["labels"] for f in features])

        # usa tokenizer.pad per input_ids + attention_mask
        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # pad manuale per e1/e2_mask alla stessa lunghezza del batch["input_ids"]
        max_len = batch["input_ids"].shape[1]

        def pad_1d(x, pad_value=0):
            # x: tensor [seq_len]
            if x.shape[0] == max_len:
                return x
            out = torch.full((max_len,), pad_value, dtype=x.dtype)
            out[: x.shape[0]] = x
            return out

        e1 = torch.stack([pad_1d(f["e1_mask"]) for f in features])
        e2 = torch.stack([pad_1d(f["e2_mask"]) for f in features])

        batch["e1_mask"] = e1
        batch["e2_mask"] = e2
        batch["labels"] = labels
        return batch


In [17]:
WINDOW_CHARS = 300
# ---- CONFIG CACHE PATHS ----
CACHE_DIR = os.path.join(output_model_dir, "cache_tok")
os.makedirs(CACHE_DIR, exist_ok=True)

TRAIN_CACHE = os.path.join(CACHE_DIR, f"train_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")
DEV_CACHE   = os.path.join(CACHE_DIR, f"dev_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")

class TensorREDataset(Dataset):
    """Custom dataset for Relation Extraction."""
    
    def __init__(self, tensor_dict):
        self.td = tensor_dict
        self.n = self.td["input_ids"].shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return {
            "input_ids": self.td["input_ids"][idx],
            "attention_mask": self.td["attention_mask"][idx],
            "e1_mask": self.td["e1_mask"][idx],
            "e2_mask": self.td["e2_mask"][idx],
            "labels": self.td["labels"][idx],
        }

from torch.utils.data import Dataset

class ListREDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return self.items[idx]

print("Dataset class defined")

def pretokenize_examples_dynamic(
    examples,
    tokenizer,
    e1_token_id,
    e2_token_id,
    max_length=512,
    window_chars=300,
    cache_path=None,
    verbose_every=5000,
):
    if cache_path is not None and os.path.exists(cache_path):
        print(f"[cache] Loading dynamic tokenized dataset from: {cache_path}")
        payload = torch.load(cache_path, map_location="cpu")
        return payload["items"], payload.get("skipped", [])

    print("[cache] Building dynamic tokenized items... (runs once)")
    items = []
    skipped = []

    for i, ex in enumerate(tqdm(examples, desc="Pre-tokenizing(dyn)", total=len(examples))):
        out = None
        try:
            out = tokenize_re_example(
                ex,
                tokenizer,
                e1_token_id,
                e2_token_id,
                max_length=max_length,
                window_chars=window_chars,
                fallback_to_fulltext=True,
            )
        except Exception as e:
            skipped.append((ex.get("pmid"), f"exception:{type(e).__name__}:{str(e)[:120]}"))
            continue

        if out is None:
            skipped.append((ex.get("pmid"), "tokenize_returned_None"))
            continue

        # safety: markers must exist
        if out["e1_mask"].sum().item() != 1 or out["e2_mask"].sum().item() != 1:
            skipped.append((ex.get("pmid"), f"bad_markers_e1={out['e1_mask'].sum().item()}_e2={out['e2_mask'].sum().item()}"))
            continue

        # ✅ IMPORTANT: out now contains variable-length tensors
        items.append({
            "input_ids": out["input_ids"].to(torch.int64),
            "attention_mask": out["attention_mask"].to(torch.int64),
            "e1_mask": out["e1_mask"].to(torch.int64),
            "e2_mask": out["e2_mask"].to(torch.int64),
            "labels": out["labels"].to(torch.int64),
        })

        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"  ...processed {i+1}/{len(examples)} | kept={len(items)} | skipped={len(skipped)}")

    print(f"[cache] Done. kept={len(items)} / {len(examples)} | skipped={len(skipped)}")

    if cache_path is not None:
        torch.save({"items": items, "skipped": skipped}, cache_path)
        print(f"[cache] Saved dynamic tokenized dataset to: {cache_path}")

        skip_txt = cache_path.replace(".pt", "_skipped.txt")
        with open(skip_txt, "w", encoding="utf-8") as f:
            for pmid, reason in skipped:
                f.write(f"{pmid}\t{reason}\n")
        print(f"[cache] Saved skipped list to: {skip_txt}")

    return items, skipped


Dataset class defined


In [18]:
WINDOW_CHARS=300

train_items, train_skipped = pretokenize_examples_dynamic(
    train_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=TRAIN_CACHE
)

dev_items, dev_skipped = pretokenize_examples_dynamic(
    dev_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=DEV_CACHE
)

train_dataset = ListREDataset(train_items)
dev_dataset   = ListREDataset(dev_items)

collator = REDataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


print("FAST datasets ready!")


[cache] Loading dynamic tokenized dataset from: models/bert_biomedbert_re\cache_tok\train_dyn_maxlen512_win300.pt
[cache] Loading dynamic tokenized dataset from: models/bert_biomedbert_re\cache_tok\dev_dyn_maxlen512_win300.pt
FAST datasets ready!


## Initialize Model

In [19]:
# Initialize model
print("Initializing BERT RE model...")
model = BertForREWithEntityMarkers(model_name, num_labels=len(RELATION_LABELS))

# Resize token embeddings to account for new special tokens
model.bert.resize_token_embeddings(len(tokenizer))

print(f"Model initialized")
print(f"  Number of labels: {model.num_labels}")
print(f"  Hidden size: {model.bert.config.hidden_size}")

Initializing BERT RE model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 277.79it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

Model initialized
  Number of labels: 18
  Hidden size: 768


## Custom Trainer for Entity Marker Model

In [20]:
import os
import torch
from transformers import Trainer

class RETrainer(Trainer):
    """Custom Trainer that handles entity marker masks + safe saving on Windows."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    # ---- CRITICAL PATCH: override _save to avoid safetensors ----
    def _save(self, output_dir: str, state_dict=None):
        os.makedirs(output_dir, exist_ok=True)

        if state_dict is None:
            state_dict = self.model.state_dict()

        # make tensors contiguous (extra-safe)
        for k, v in state_dict.items():
            if isinstance(v, torch.Tensor) and not v.is_contiguous():
                state_dict[k] = v.contiguous()

        # save as classic pytorch bin
        torch.save(state_dict, os.path.join(output_dir, "pytorch_model.bin"))

        # (optional but nice) also save training args
        torch.save(self.args, os.path.join(output_dir, "training_args.bin"))

## Configure Training Arguments

In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    learning_rate=2e-5,
    warmup_ratio=0.06,                 # aiuta stabilità all'inizio
    lr_scheduler_type="linear",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,     # batch effettivo 32 (se regge)
    max_grad_norm=1.0,                 # clipping

    num_train_epochs=3,
    weight_decay=0.01,

    eval_strategy="steps",       # meglio che "epoch" con dataset grosso
    eval_steps=5000,                   # ~10 eval per epoca (20k step/epoca)
    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,


    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    disable_tqdm=False,
    logging_steps=10,

    fp16=torch.cuda.is_available(),
    tf32=True,    # accelera su GPU recenti
    dataloader_num_workers=0,          # velocizza input pipeline (su Windows ok)
    dataloader_pin_memory=False,

    seed=SEED,
    report_to="none"
)

print("Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training configuration ready
  Batch size: 16
  Epochs: 3
  Learning rate: 2e-05


## Train Model

**Note:** This cell might take several minutes to hours depending on dataset size and hardware.

**Hyperparameters to experiment with:**
- `NEGATIVE_SAMPLE_MULTIPLIER`: Try 1, 2, 3, 5
- `learning_rate`: Try 1e-5, 2e-5, 3e-5
- `num_train_epochs`: Try 3, 5, 10
- `per_device_train_batch_size`: Adjust based on GPU memory
- Different pretrained models: "allenai/scibert_scivocab_uncased", "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

In [22]:
# Initialize trainer
print("Initializing Trainer...")
trainer = RETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=collator,
)

print("Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

Initializing Trainer...
Trainer initialized
  Training samples: 321438
  Evaluation samples: 6611


In [23]:
# Start training
print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


Step,Training Loss,Validation Loss
5000,0.152073,0.256718
10000,0.203304,0.287071
15000,0.176164,0.308648
20000,0.187848,0.269312
25000,0.073488,0.312277
30000,0.130334,0.325501



TRAINING COMPLETED!
Training time: 182.04 minutes


## Save Trained Model

In [24]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

# Save label mappings
with open(os.path.join(output_model_dir, 'label_mappings.json'), 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)

print(f"Model saved to: {output_model_dir}")

Saving trained model...
Model saved to: models/bert_biomedbert_re


## Load Model for Inference

In [56]:
import os
import torch

def find_last_checkpoint(root_dir: str) -> str:
    ckpts = [d for d in os.listdir(root_dir) if d.startswith("checkpoint-")]
    if not ckpts:
        return root_dir
    ckpts.sort(key=lambda x: int(x.split("-")[1]))
    return os.path.join(root_dir, ckpts[-1])

print("Loading trained model for inference...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) tokenizer (quello salvato, così include i token speciali)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir, use_fast=True)
e1_token_id = inference_tokenizer.convert_tokens_to_ids("[E1]")
e2_token_id = inference_tokenizer.convert_tokens_to_ids("[E2]")

# 2) scegli da dove caricare
load_dir = find_last_checkpoint(output_model_dir)
state_path = os.path.join(load_dir, "pytorch_model.bin")
print("Loading from:", state_path)

# 3) ricrea architettura + resize embeddings prima del load
inference_model = BertForREWithEntityMarkers(model_name, num_labels=len(RELATION_LABELS))
inference_model.bert.resize_token_embeddings(len(inference_tokenizer))

# 4) load state_dict
state_dict = torch.load(state_path, map_location="cpu")
inference_model.load_state_dict(state_dict)

# 5) to device
inference_model.to(device)
inference_model.eval()

print("Model loaded successfully")
print("  Device:", device)
print("  Last checkpoint:", load_dir)

Loading trained model for inference...
Loading from: models/bert_biomedbert_re\checkpoint-30135\pytorch_model.bin


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 511.32it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

Model loaded successfully
  Device: cuda
  Last checkpoint: models/bert_biomedbert_re\checkpoint-30135


## Inference

## Inference configuration

This section defines inference-time parameters used for batching and truncation control.
We use:
- `BATCH_SIZE` to control GPU/CPU throughput
- `MAX_LENGTH` for tokenizer truncation (512 for BERT)
- `WINDOW_CHARS` to build a local context window around entity mentions (prevents truncation of entity markers)
- `MIN_PROB` and `MARGIN` for margin-based decoding (used for a quick single-run inference, and as candidates for tuning)

In [57]:
BATCH_SIZE = 16
MAX_LENGTH = 512
WINDOW_CHARS = 300

# Margin decoding defaults (used for quick inference runs, not for grid search)
MIN_PROB = 0.25
MARGIN = 0.20

## Windowed inference text construction (same strategy as training)

We insert entity markers `[E1]...[/E1]` and `[E2]...[/E2]` around the mentions.
To make inference consistent with training and avoid truncation issues, we:
1) extract a character window around the two mentions
2) shift offsets to the window coordinates
3) insert entity markers inside the window

This greatly stabilizes inference compared to inserting markers in the full document and truncating.

In [58]:
def build_marked_text_for_inference(text, subj, obj, window_chars=300):
    w_text, w_subj, w_obj = build_window_around_entities(text, subj, obj, window_chars=window_chars)
    return insert_entity_markers(w_text, w_subj, w_obj)

## Margin-based decoding (single-pass on a list of candidate pairs)

Instead of using a single global threshold on the top predicted class,
we apply **margin-based decoding**:

Keep the prediction only if:
- `p_best >= MIN_PROB`
- `(p_best - p_no_relation) >= MARGIN`

This reduces false positives where the model is also uncertain and still assigns a relation label.

In [60]:
@torch.no_grad()
def predict_with_margin_on_examples(
    model,
    tokenizer,
    examples,              # list of (full_text, subj, obj)
    e1_token_id,
    e2_token_id,
    device,
    batch_size=16,
    max_length=512,
    window_chars=300,
    min_prob=0.25,
    margin=0.20,
):
    """
    Returns list aligned to examples:
      (keep: bool, pred_id: int, p_best: float, p_no: float)
    where pred_id is the best non-'no relation' label.
    """
    model.eval()
    out = []

    marked_texts = [
        build_marked_text_for_inference(text, subj, obj, window_chars=window_chars)
        for (text, subj, obj) in examples
    ]

    for start in range(0, len(marked_texts), batch_size):
        batch_texts = marked_texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        e1_mask = (input_ids == e1_token_id).long()
        e2_mask = (input_ids == e2_token_id).long()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            e1_mask=e1_mask,
            e2_mask=e2_mask
        )["logits"]

        probs = torch.softmax(logits, dim=-1)

        # assumes "no relation" label id == 0
        p_no = probs[:, 0]
        probs_non = probs[:, 1:]
        best_non_id = torch.argmax(probs_non, dim=-1) + 1
        p_best = probs[torch.arange(probs.size(0)), best_non_id]

        keep = (p_best >= min_prob) & ((p_best - p_no) >= margin)

        for k, pid, pb, pno in zip(keep.tolist(), best_non_id.tolist(), p_best.tolist(), p_no.tolist()):
            out.append((bool(k), int(pid), float(pb), float(pno)))

    return out

## Dev-only tuning (no retraining): cache scores once, then grid-search decoding
To get the maximum performance, we:
1) run the model **once** on the dev set and cache, for each candidate pair:
   - `pred_id`: best non-`no relation` label
   - `p_best`: probability of that label
   - `p_no`: probability of `no relation`
   - `dist`: character distance between the two mentions
2) perform a grid search over decoding parameters using the cached values:
   - `MAX_PAIR_CHARS` ∈ {300, 400, 500}
   - `MIN_PROB` ∈ {0.25, 0.35, 0.45}
   - `MARGIN` ∈ {0.10, 0.15, 0.20}

This is **pure inference tuning** (no training, no backprop).

### Utility functions for evaluation (micro-F1 + per-predicate PRF1)

We evaluate predictions against dev gold relations using:
- global micro-precision / micro-recall / micro-F1
- per-predicate micro PRF1 to find weak relation types

In [61]:
import itertools
from collections import Counter, defaultdict


# --- helpers: keys ---
def gold_tuple(r):
    return (
        norm_span(r["subject_text_span"]),
        norm_ent(r["subject_label"]),
        r["predicate"].strip(),
        norm_span(r["object_text_span"]),
        norm_ent(r["object_label"]),
    )

def build_gold_maps(dev_data):
    """
    Returns:
      gold_by_doc[pmid] = set of (s_text,s_lab,pred,o_text,o_lab)
      gold_by_pred[pred] = set of tuples over all docs
    """
    gold_by_doc = {}
    gold_by_pred = defaultdict(set)

    for pmid, art in dev_data.items():
        s = set()
        for r in art.get("mention_level_relations", []):
            pred = r["predicate"].strip()
            if pred not in LEGAL_RELATION_LABELS:
                continue
            t = gold_tuple(r)
            s.add(t)
            gold_by_pred[pred].add(t)
        gold_by_doc[str(pmid)] = s
    return gold_by_doc, gold_by_pred


def micro_scores(gold_by_doc, pred_by_doc):
    tp = fp = fn = 0
    for pmid, g in gold_by_doc.items():
        p = pred_by_doc.get(pmid, set())
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0
    return {"P": prec, "R": rec, "F1": f1, "TP": tp, "FP": fp, "FN": fn}


def per_predicate_scores(gold_by_doc, pred_by_doc):
    """
    Returns per-predicate PRF1 over all docs (micro per label).
    """
    # gather gold and pred by predicate
    gold_pred = defaultdict(set)
    pred_pred = defaultdict(set)

    for pmid, gset in gold_by_doc.items():
        for t in gset:
            gold_pred[t[2]].add(t)

    for pmid, pset in pred_by_doc.items():
        for t in pset:
            pred_pred[t[2]].add(t)

    out = {}
    for pred in sorted(set(gold_pred.keys()) | set(pred_pred.keys())):
        g = gold_pred.get(pred, set())
        p = pred_pred.get(pred, set())
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0
        out[pred] = {"P": prec, "R": rec, "F1": f1, "TP": tp, "FP": fp, "FN": fn, "Gold": len(g), "Pred": len(p)}
    return out



### One-pass scoring cache on dev (forward pass only once)

This function builds candidate mention pairs per document and caches model scores:
- candidate pairs are limited to `legal_pairs` (schema constraints)
- we compute and store `dist` between mentions
- we compute `p_best`, `p_no`, and `pred_id` from the model

The resulting `dev_cache` can be decoded multiple times with different thresholds without rerunning the model.

In [62]:

@torch.no_grad()
def cache_dev_pair_scores_once(
        model,
        tokenizer,
        dev_data,
        legal_pairs,
        e1_token_id,
        e2_token_id,
        device,
        batch_size=16,
        max_length=512,
        window_chars=300,
):
    """
    One forward pass on dev:
    cache[pmid] = list of rows:
      {
        "k": (s_text,s_lab,o_text,o_lab),
        "dist": int,
        "pred_id": int (best non-no-rel),
        "p_best": float,
        "p_no": float
      }
    """
    model.eval()
    cache = {}

    for pmid, article in tqdm(dev_data.items(), desc="Caching dev pair scores (ONE pass)"):
        title = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abstract_offset = create_full_text_with_offsets(title, abstract)

        # entities normalized
        entities = article["entities"]
        adjusted_entities = [
            {**adjust_entity_positions(e, abstract_offset),
             "label": norm_ent(e["label"]),
             "text_span": norm_span(e["text_span"])}
            for e in entities
        ]

        pair_examples = []
        pair_keys = []
        pair_dists = []

        for i, subj in enumerate(adjusted_entities):
            for j, obj in enumerate(adjusted_entities):
                if i == j:
                    continue
                type_pair = (subj["label"], obj["label"])
                if type_pair not in legal_pairs:
                    continue

                k = (subj["text_span"], subj["label"], obj["text_span"], obj["label"])
                dist = abs(subj["start_idx"] - obj["start_idx"])

                pair_examples.append((full_text, subj, obj))
                pair_keys.append(k)
                pair_dists.append(dist)

        if not pair_examples:
            cache[str(pmid)] = []
            continue

        # build windowed marked texts
        marked_texts = [
            build_marked_text_for_inference(text, subj, obj, window_chars=window_chars)
            for (text, subj, obj) in pair_examples
        ]

        scored = []
        for start in range(0, len(marked_texts), batch_size):
            batch_texts = marked_texts[start:start + batch_size]
            enc = tokenizer(batch_texts, truncation=True, max_length=max_length, padding=True, return_tensors="pt")

            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            e1_mask = (input_ids == e1_token_id).long()
            e2_mask = (input_ids == e2_token_id).long()

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                e1_mask=e1_mask,
                e2_mask=e2_mask
            )["logits"]

            probs = torch.softmax(logits, dim=-1)

            p_no = probs[:, 0]
            probs_non = probs[:, 1:]
            best_non_id = torch.argmax(probs_non, dim=-1) + 1
            p_best = probs[torch.arange(probs.size(0)), best_non_id]

            for pid, pb, pn in zip(best_non_id.tolist(), p_best.tolist(), p_no.tolist()):
                scored.append((int(pid), float(pb), float(pn)))

        rows = []
        for k, dist, (pred_id, p_best, p_no) in zip(pair_keys, pair_dists, scored):
            rows.append({
                "k": k,
                "dist": dist,
                "pred_id": pred_id,
                "p_best": p_best,
                "p_no": p_no,
                "margin_val": (p_best - p_no),
            })

        cache[str(pmid)] = rows

    return cache


## Decode a document from cache (distance filter + margin decoding + dedup)

Given cached scores for a document, we apply:
1) distance filter: `dist <= MAX_PAIR_CHARS`
2) margin decoding: `p_best >= MIN_PROB` and `p_best - p_no >= MARGIN`
3) deduplication: keep the best scoring predicate for each mention pair key

Outputs:
- a set of relation tuples for evaluation
- `best_for_pair` dict used to export submission JSON

In [63]:

def decode_doc_from_cache(rows, id2label, max_pair_chars, min_prob, margin):
    """
    Decode one doc from cached rows -> set of (s_text,s_lab,pred,o_text,o_lab)
    with dedup by (s_text,s_lab,o_text,o_lab) keeping best p_best.
    """
    best_for_pair = {}
    for r in rows:
        if r["dist"] > max_pair_chars:
            continue
        if r["p_best"] < min_prob:
            continue
        if (r["p_best"] - r["p_no"]) < margin:
            continue

        pred = id2label[r["pred_id"]]
        if pred not in LEGAL_RELATION_LABELS:
            continue

        k = r["k"]  # (s_text,s_lab,o_text,o_lab)
        prev = best_for_pair.get(k)
        if (prev is None) or (r["p_best"] > prev[1]):
            best_for_pair[k] = (pred, r["p_best"])

    out_set = set()
    for (s_text, s_lab, o_text, o_lab), (pred, score) in best_for_pair.items():
        if s_lab not in LEGAL_ENTITY_LABELS or o_lab not in LEGAL_ENTITY_LABELS:
            continue
        out_set.add((s_text, s_lab, pred, o_text, o_lab))
    return out_set, best_for_pair


def build_predictions_json_from_best(best_for_pair):
    """
    Convert best_for_pair dict -> list for submission json.
    best_for_pair: (s_text,s_lab,o_text,o_lab) -> (pred, score)
    """
    mention_level = []
    for (s_text, s_lab, o_text, o_lab), (pred, score) in sorted(best_for_pair.items()):
        if s_lab not in LEGAL_ENTITY_LABELS or o_lab not in LEGAL_ENTITY_LABELS:
            continue
        mention_level.append({
            "subject_text_span": s_text,
            "subject_label": s_lab,
            "predicate": pred,
            "object_text_span": o_text,
            "object_label": o_lab
        })
    return mention_level

## Run dev cache + grid search (no retraining)

This cell:
1) builds the dev gold map
2) computes `dev_cache` with a single model pass
3) performs grid search over decoding parameters
4) selects the best setting by micro-F1
5) generates and saves the final dev predictions JSON using the best setting

In [64]:
# --- build gold ---
gold_by_doc, _ = build_gold_maps(dev_data)

# --- cache scores ONCE ---
dev_cache = cache_dev_pair_scores_once(
    inference_model,
    inference_tokenizer,
    dev_data,
    legal_pairs,
    e1_token_id,
    e2_token_id,
    device,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    window_chars=WINDOW_CHARS,
)

# --- grid search params (requested) ---
MAX_PAIR_CHARS_GRID = [300, 400, 500]
MIN_PROB_GRID = [0.25, 0.35, 0.45]
MARGIN_GRID = [0.10, 0.15, 0.20]

grid_results = []

for max_chars, min_prob, margin in itertools.product(MAX_PAIR_CHARS_GRID, MIN_PROB_GRID, MARGIN_GRID):
    pred_by_doc = {}
    for pmid, rows in dev_cache.items():
        pred_set, _best = decode_doc_from_cache(rows, id2label, max_chars, min_prob, margin)
        pred_by_doc[pmid] = pred_set

    ms = micro_scores(gold_by_doc, pred_by_doc)
    grid_results.append({
        "MAX_PAIR_CHARS": max_chars,
        "MIN_PROB": min_prob,
        "MARGIN": margin,
        **ms
    })

grid_results = sorted(grid_results, key=lambda d: d["F1"], reverse=True)

print("\nTop 10 settings by micro-F1:")
for r in grid_results[:10]:
    print(r)

best = grid_results[0]
print("\nBEST:", best)

# --- decode with best params and build submission-style predictions json ---
BEST_MAX = best["MAX_PAIR_CHARS"]
BEST_MINP = best["MIN_PROB"]
BEST_MARG = best["MARGIN"]

predictions = {}
pred_by_doc_best = {}  # for analysis

for pmid, rows in dev_cache.items():
    pred_set, best_for_pair = decode_doc_from_cache(rows, id2label, BEST_MAX, BEST_MINP, BEST_MARG)
    pred_by_doc_best[pmid] = pred_set
    predictions[str(pmid)] = {"mention_level_relations": build_predictions_json_from_best(best_for_pair)}

total_relations = sum(len(p["mention_level_relations"]) for p in predictions.values())
print(f"\nDocs: {len(predictions)} | Total mention-level relations (kept): {total_relations}")

# save with params in name
out_path = f"predictions/bert_re_grid_best_chars{BEST_MAX}_minp{BEST_MINP}_m{BEST_MARG}.json"
os.makedirs("predictions", exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("Saved:", out_path)

Caching dev pair scores (ONE pass): 100%|██████████| 80/80 [05:42<00:00,  4.29s/it]



Top 10 settings by micro-F1:
{'MAX_PAIR_CHARS': 300, 'MIN_PROB': 0.25, 'MARGIN': 0.2, 'P': 0.465658475110271, 'R': 0.6488147497805092, 'F1': 0.5421863536316948, 'TP': 739, 'FP': 848, 'FN': 400}
{'MAX_PAIR_CHARS': 300, 'MIN_PROB': 0.35, 'MARGIN': 0.2, 'P': 0.465658475110271, 'R': 0.6488147497805092, 'F1': 0.5421863536316948, 'TP': 739, 'FP': 848, 'FN': 400}
{'MAX_PAIR_CHARS': 300, 'MIN_PROB': 0.45, 'MARGIN': 0.2, 'P': 0.465658475110271, 'R': 0.6488147497805092, 'F1': 0.5421863536316948, 'TP': 739, 'FP': 848, 'FN': 400}
{'MAX_PAIR_CHARS': 300, 'MIN_PROB': 0.45, 'MARGIN': 0.15, 'P': 0.46068111455108357, 'R': 0.6532045654082529, 'F1': 0.5403050108932462, 'TP': 744, 'FP': 871, 'FN': 395}
{'MAX_PAIR_CHARS': 300, 'MIN_PROB': 0.25, 'MARGIN': 0.15, 'P': 0.4603960396039604, 'R': 0.6532045654082529, 'F1': 0.5401088929219601, 'TP': 744, 'FP': 872, 'FN': 395}
{'MAX_PAIR_CHARS': 300, 'MIN_PROB': 0.35, 'MARGIN': 0.15, 'P': 0.4603960396039604, 'R': 0.6532045654082529, 'F1': 0.5401088929219601, 'TP': 

## Dev-only tuning: predicate-specific margins (no retraining)

A single global margin works well overall, but different predicates behave differently:
- some predicates are over-predicted (many false positives) → need a stricter margin
- some predicates are under-predicted (many false negatives) → need a looser margin

Here we tune a *separate margin threshold per predicate* using the dev set.
This is still inference-only tuning: we reuse the cached scores (`dev_cache`) and do not retrain the model.

In [65]:
MARGINS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

# Use the best global distance constraint found by grid search
MAXC = BEST_MAX  # e.g., 300
MINP = 0.0  # keep min prob inactive; margin will drive the decision


def decode_doc_pred_specific(rows, id2label, max_pair_chars, min_prob, margin_by_pred, default_margin=0.20):
    """
    Decode a doc using predicate-specific margins.
    Dedup by (s_text,s_lab,o_text,o_lab) keeping best p_best.
    Returns a set of (s_text,s_lab,pred,o_text,o_lab).
    """
    best_for_pair = {}
    for r in rows:
        if r["dist"] > max_pair_chars:
            continue
        if r["p_best"] < min_prob:
            continue

        pred = id2label[r["pred_id"]]
        if pred not in LEGAL_RELATION_LABELS:
            continue

        m = margin_by_pred.get(pred, default_margin)
        if (r["p_best"] - r["p_no"]) < m:
            continue

        k = r["k"]
        prev = best_for_pair.get(k)
        if (prev is None) or (r["p_best"] > prev[1]):
            best_for_pair[k] = (pred, r["p_best"])

    out = set()
    for (s_text, s_lab, o_text, o_lab), (pred, score) in best_for_pair.items():
        out.add((s_text, s_lab, pred, o_text, o_lab))
    return out


# Compute best margin independently for each predicate (micro-F1 on that predicate)
best_margin_by_pred = {}

gold_preds = sorted({t[2] for g in gold_by_doc.values() for t in g})

for pred in gold_preds:
    best = None

    for m in MARGINS:
        # predict ONLY this predicate at margin m (others suppressed with default_margin=1.0)
        pred_by_doc = {}
        for pmid, rows in dev_cache.items():
            margin_by_pred_tmp = {pred: m}
            pred_set = decode_doc_pred_specific(
                rows, id2label, MAXC, MINP, margin_by_pred_tmp, default_margin=1.0
            )
            pred_set = {t for t in pred_set if t[2] == pred}
            pred_by_doc[pmid] = pred_set

        # micro scores for this predicate only
        tp = fp = fn = 0
        for pmid, g in gold_by_doc.items():
            g_pred = {t for t in g if t[2] == pred}
            p_pred = pred_by_doc.get(pmid, set())
            tp += len(g_pred & p_pred)
            fp += len(p_pred - g_pred)
            fn += len(g_pred - p_pred)

        P = tp / (tp + fp) if (tp + fp) else 0.0
        R = tp / (tp + fn) if (tp + fn) else 0.0
        F1 = (2 * P * R) / (P + R) if (P + R) else 0.0

        if (best is None) or (F1 > best["F1"]):
            best = {"pred": pred, "margin": m, "P": P, "R": R, "F1": F1, "TP": tp, "FP": fp, "FN": fn}

    best_margin_by_pred[pred] = best

print("Best margin per predicate (dev):")
for pred in sorted(best_margin_by_pred.keys()):
    b = best_margin_by_pred[pred]
    print(f"{pred:15s} margin={b['margin']:.2f} F1={b['F1']:.3f} (P={b['P']:.3f} R={b['R']:.3f})")

Best margin per predicate (dev):
administered    margin=0.15 F1=0.252 (P=0.419 R=0.181)
affect          margin=0.30 F1=0.536 (P=0.484 R=0.600)
change abundance margin=0.30 F1=0.300 (P=0.205 R=0.562)
change effect   margin=0.05 F1=0.656 (P=0.667 R=0.645)
change expression margin=0.05 F1=0.727 (P=1.000 R=0.571)
compared to     margin=0.05 F1=0.000 (P=0.000 R=0.000)
impact          margin=0.10 F1=0.500 (P=0.407 R=0.649)
influence       margin=0.30 F1=0.552 (P=0.469 R=0.669)
interact        margin=0.30 F1=0.353 (P=0.245 R=0.632)
is a            margin=0.30 F1=0.626 (P=0.488 R=0.872)
is linked to    margin=0.15 F1=0.619 (P=0.525 R=0.755)
located in      margin=0.30 F1=0.553 (P=0.464 R=0.684)
part of         margin=0.20 F1=0.312 (P=0.255 R=0.400)
produced by     margin=0.15 F1=0.400 (P=0.250 R=1.000)
strike          margin=0.30 F1=0.632 (P=0.529 R=0.783)
target          margin=0.10 F1=0.741 (P=0.719 R=0.764)
used by         margin=0.25 F1=0.588 (P=0.515 R=0.685)


## Decode using predicate-specific margins and export predictions

Using the tuned margin for each predicate, we decode the cached pair scores again:
- apply the best global distance constraint (`MAXC`)
- apply a predicate-dependent margin threshold
- deduplicate by mention pair and keep the best-scoring predicate

Finally, we compute global micro metrics and export the dev predictions JSON.

In [85]:
margin_by_pred = {p: best_margin_by_pred[p]["margin"] for p in best_margin_by_pred}

pred_by_doc_ps = {}
predictions_ps = {}

for pmid, rows in dev_cache.items():
    # 1) decode set for metrics (predicate-specific margins)
    pred_set = decode_doc_pred_specific(
        rows, id2label, MAXC, 0.0, margin_by_pred, default_margin=0.20
    )
    pred_by_doc_ps[pmid] = pred_set

    # 2) build submission json with dedup best p_best per pair
    best_for_pair = {}
    for r in rows:
        if r["dist"] > MAXC:
            continue
        pred = id2label[r["pred_id"]]
        if pred not in LEGAL_RELATION_LABELS:
            continue

        m = margin_by_pred.get(pred, 0.20)
        if (r["p_best"] - r["p_no"]) < m:
            continue

        k = r["k"]
        prev = best_for_pair.get(k)
        if (prev is None) or (r["p_best"] > prev[1]):
            best_for_pair[k] = (pred, r["p_best"])

    predictions_ps[str(pmid)] = {"mention_level_relations": build_predictions_json_from_best(best_for_pair)}


# TODO
# 1) Choose what to analyze (DO THIS FIRST)
predictions_to_analyze = predictions_ps
pred_by_doc_to_analyze = pred_by_doc_ps

print("\n== Global micro metrics (selected decoding) ==")
print(micro_scores(gold_by_doc, pred_by_doc_to_analyze))

per_pred = per_predicate_scores(gold_by_doc, pred_by_doc_to_analyze)
out_path = "predictions/old/bert_re_dev_predspec_margin.json"
os.makedirs("predictions", exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(predictions_to_analyze, f, ensure_ascii=False, indent=2)
print("Saved:", out_path)


== Global micro metrics (selected decoding) ==
{'P': 0.4753678822776711, 'R': 0.6523266022827041, 'F1': 0.5499629903774981, 'TP': 743, 'FP': 820, 'FN': 396}
Saved: predictions/bert_re_dev_predspec_margin.json


## Report global metrics and per-predicate performance (dev)

After decoding, we compute:
- global micro precision/recall/F1
- per-predicate PRF1 to identify which relations are over-predicted (many FP) or under-predicted (many FN)

This analysis guides the next inference-only improvements (e.g., predicate-specific thresholds)
or future training changes (hard negatives, class weights, etc.).

In [86]:
print("== Global micro metrics (selected decoding) ==")
print(micro_scores(gold_by_doc, pred_by_doc_to_analyze))

per_pred = per_predicate_scores(gold_by_doc, pred_by_doc_to_analyze)
worst = sorted([(p, s) for p, s in per_pred.items() if s["Gold"] > 0], key=lambda x: x[1]["F1"])
print("\n== Worst predicates by F1 (best decoding) ==")
for p, s in worst[:8]:
    print(p, s)

bestp = sorted([(p, s) for p, s in per_pred.items() if s["Gold"] > 0], key=lambda x: x[1]["F1"], reverse=True)
print("\n== Best predicates by F1 (best decoding) ==")
for p, s in bestp[:8]:
    print(p, s)

== Global micro metrics (selected decoding) ==
{'P': 0.4753678822776711, 'R': 0.6523266022827041, 'F1': 0.5499629903774981, 'TP': 743, 'FP': 820, 'FN': 396}

== Worst predicates by F1 (best decoding) ==
compared to {'P': 0.0, 'R': 0.0, 'F1': 0.0, 'TP': 0, 'FP': 0, 'FN': 4, 'Gold': 4, 'Pred': 0}
administered {'P': 0.41935483870967744, 'R': 0.18055555555555555, 'F1': 0.25242718446601947, 'TP': 13, 'FP': 18, 'FN': 59, 'Gold': 72, 'Pred': 31}
change abundance {'P': 0.20454545454545456, 'R': 0.5625, 'F1': 0.3, 'TP': 9, 'FP': 35, 'FN': 7, 'Gold': 16, 'Pred': 44}
part of {'P': 0.26666666666666666, 'R': 0.4, 'F1': 0.32, 'TP': 12, 'FP': 33, 'FN': 18, 'Gold': 30, 'Pred': 45}
interact {'P': 0.24489795918367346, 'R': 0.631578947368421, 'F1': 0.3529411764705882, 'TP': 24, 'FP': 74, 'FN': 14, 'Gold': 38, 'Pred': 98}
produced by {'P': 0.23809523809523808, 'R': 1.0, 'F1': 0.3846153846153846, 'TP': 5, 'FP': 16, 'FN': 0, 'Gold': 5, 'Pred': 21}
impact {'P': 0.42045454545454547, 'R': 0.6491228070175439, '

## Example Predictions

In [87]:
# Show example predictions
print("Example Predictions (Mention-level RE):\n")

sample_pmids = list(dev_data.keys())[:5]

for pmid in sample_pmids:
    article = dev_data[pmid]
    pred_doc = predictions_to_analyze[str(pmid)]

    gold_rels = article.get("mention_level_relations", [])
    pred_rels = pred_doc.get("mention_level_relations", [])

    print(f"Document PMID: {pmid}")
    print(f"Title: {article['metadata']['title'][:100]}...")
    print(f"\nNumber of entities: {len(article.get('entities', []))}")
    print(f"Number of GOLD mention relations: {len(gold_rels)}")
    print(f"Number of PRED mention relations: {len(pred_rels)}")

    print("\nSample predicted mention relations:")
    for r in pred_rels[:5]:
        print(f"  ({r['subject_text_span']} [{r['subject_label']}]) "
              f"--[{r['predicate']}]--> "
              f"({r['object_text_span']} [{r['object_label']}])")

    print("-" * 80)
    print()

Example Predictions (Mention-level RE):

Document PMID: 35766370
Title: Orthopedic Surgery Causes Gut Microbiome Dysbiosis and Intestinal Barrier Dysfunction in Prodromal A...

Number of entities: 45
Number of GOLD mention relations: 43
Number of PRED mention relations: 46

Sample predicted mention relations:
  (Gut Microbiome Dysbiosis [DDF]) --[target]--> (Prodromal Alzheimer Disease Patients [human])
  (Homeostatic disturbances [DDF]) --[target]--> (elderly patients [human])
  (Homeostatic disturbances [DDF]) --[change abundance]--> (gut microbiota [microbiome])
  (Intestinal Barrier Dysfunction [DDF]) --[target]--> (Prodromal Alzheimer Disease Patients [human])
  (NC [DDF]) --[target]--> (elderly patients [human])
--------------------------------------------------------------------------------

Document PMID: 34912029
Title: Cadmium exposure modulates the gut-liver axis in an Alzheimer's disease mouse model....

Number of entities: 24
Number of GOLD mention relations: 15
Number of 

In [ ]:
%%sql


## Relation distribution comparison (Gold vs Predicted)

This table compares how frequently each predicate appears in:
- gold dev relations
- predicted relations

Large mismatches (e.g., too many `located in`) often indicate:
- overly permissive thresholds for that predicate
- candidate pair explosion for that predicate
- the model confusing that predicate with others

In [88]:
from collections import Counter

pred_predicate_counts = Counter()
for pmid, pred_doc in predictions_to_analyze.items():
    for r in pred_doc.get("mention_level_relations", []):
        pred_predicate_counts[r["predicate"]] += 1

gold_predicate_counts = Counter()
for pmid, article in dev_data.items():
    for r in article.get("mention_level_relations", []):
        gold_predicate_counts[r["predicate"]] += 1

print("Mention-level Relation Distribution by Predicate:")
print("=" * 60)
print(f"{'Predicate':<30} {'Gold':<10} {'Predicted':<10}")
print("-" * 60)

all_predicates = set(gold_predicate_counts.keys()) | set(pred_predicate_counts.keys())
for predicate in sorted(all_predicates):
    print(f"{predicate:<30} {gold_predicate_counts[predicate]:<10} {pred_predicate_counts[predicate]:<10}")

print("-" * 60)
print(f"{'TOTAL':<30} {sum(gold_predicate_counts.values()):<10} {sum(pred_predicate_counts.values()):<10}")


Mention-level Relation Distribution by Predicate:
Predicate                      Gold       Predicted 
------------------------------------------------------------
administered                   72         31        
affect                         125        155       
change abundance               16         44        
change effect                  31         30        
change expression              7          4         
compared to                    4          0         
impact                         57         89        
influence                      160        227       
interact                       38         98        
is a                           47         84        
is linked to                   110        158       
located in                     196        289       
part of                        30         47        
produced by                    6          24        
strike                         23         34        
target                         144       

### Error analysis: frequent FP/FN by predicate + concrete examples

We compute:
- FP: predicted relations not in gold
- FN: gold relations missing in predictions

Then we rank predicates by number of FP and FN, and print a few example triples.
This helps identify:
- predicates that are **over-predicted** (many FP → increase strictness)
- predicates that are **under-predicted** (many FN → relax thresholds or adjust candidate filtering)

In [89]:

def collect_fp_fn_examples(gold_by_doc, pred_by_doc, max_examples=30):
    fp_by_pred = defaultdict(list)
    fn_by_pred = defaultdict(list)

    for pmid, gold_set in gold_by_doc.items():
        pred_set = pred_by_doc.get(pmid, set())
        fps = list(pred_set - gold_set)
        fns = list(gold_set - pred_set)

        for t in fps:
            fp_by_pred[t[2]].append((pmid, t))
        for t in fns:
            fn_by_pred[t[2]].append((pmid, t))

    # sort predicates by count
    fp_counts = sorted(((p, len(v)) for p, v in fp_by_pred.items()), key=lambda x: x[1], reverse=True)
    fn_counts = sorted(((p, len(v)) for p, v in fn_by_pred.items()), key=lambda x: x[1], reverse=True)

    return fp_by_pred, fn_by_pred, fp_counts, fn_counts

fp_by_pred, fn_by_pred, fp_counts, fn_counts = collect_fp_fn_examples(gold_by_doc, pred_by_doc_to_analyze)
print("\n== Top FP predicates (count) ==")
for p, c in fp_counts[:10]:
    print(p, c)

print("\n== Top FN predicates (count) ==")
for p, c in fn_counts[:10]:
    print(p, c)

# show a few concrete examples for the worst FN predicate
if fn_counts:
    worst_fn_pred = fn_counts[0][0]
    print(f"\nExamples FN for predicate='{worst_fn_pred}' (up to 10):")
    for pmid, t in fn_by_pred[worst_fn_pred][:10]:
        s_text, s_lab, pred, o_text, o_lab = t
        print(f"PMID {pmid}: ({s_text} [{s_lab}]) --[{pred}]--> ({o_text} [{o_lab}])")

# show a few concrete examples for the worst FP predicate
if fp_counts:
    worst_fp_pred = fp_counts[0][0]
    print(f"\nExamples FP for predicate='{worst_fp_pred}' (up to 10):")
    for pmid, t in fp_by_pred[worst_fp_pred][:10]:
        s_text, s_lab, pred, o_text, o_lab = t
        print(f"PMID {pmid}: ({s_text} [{s_lab}]) --[{pred}]--> ({o_text} [{o_lab}])")


== Top FP predicates (count) ==
located in 155
influence 120
affect 80
is linked to 75
interact 74
impact 52
used by 47
is a 43
target 42
change abundance 35

== Top FN predicates (count) ==
located in 62
administered 59
influence 53
affect 50
target 34
is linked to 27
used by 23
impact 20
part of 18
interact 14

Examples FN for predicate='located in' (up to 10):
PMID 35766370: (inflammatory cytokines [chemical]) --[located in]--> (patients [human])
PMID 35766370: (short-chain fatty acid (SCFA)-producing bacteria [bacteria]) --[located in]--> (NC group [human])
PMID 35766370: (lipopolysaccharide [chemical]) --[located in]--> (patients [human])
PMID 35766370: (bacterial endotoxin [chemical]) --[located in]--> (patients [human])
PMID 35766370: (tight junction (TJ) protein [chemical]) --[located in]--> (patients [human])
PMID 35766370: (gram-negative bacteria [bacteria]) --[located in]--> (NC group [human])
PMID 34912029: (Cd [chemical]) --[located in]--> (ApoE3 (common allele)-KI mice [

In [90]:
print("same set object?", pred_by_doc_to_analyze is pred_by_doc_best)
print("micro(selected):", micro_scores(gold_by_doc, pred_by_doc_to_analyze))
print("micro(gridbest):", micro_scores(gold_by_doc, pred_by_doc_best))

same set object? False
micro(selected): {'P': 0.4753678822776711, 'R': 0.6523266022827041, 'F1': 0.5499629903774981, 'TP': 743, 'FP': 820, 'FN': 396}
micro(gridbest): {'P': 0.465658475110271, 'R': 0.6488147497805092, 'F1': 0.5421863536316948, 'TP': 739, 'FP': 848, 'FN': 400}
